In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("groupby")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

df_green = spark.read.parquet("data/pq/green/*/*")
df_green.createOrReplaceTempView("green")
print("Input partitions:", df_green.rdd.getNumPartitions())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 21:34:23 WARN Utils: Your hostname, DESKTOP-8DK3P4V, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/18 21:34:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 21:34:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/18 21:34:42 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.

Input partitions: 3


In [2]:
query = """
    SELECT
        date_trunc('hour', lpep_pickup_datetime) AS hour,
        PULocationID AS zone,
        SUM(total_amount) AS amount,
        COUNT(1) AS number_records
    FROM green
    WHERE lpep_pickup_datetime >= '2020-01-01'
    GROUP BY 1, 2
"""
df_green_revenue = spark.sql(query)
df_green_revenue.show()


+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-06 07:00:00| 241|            169.16|             6|
|2020-01-19 19:00:00|  41| 457.2400000000003|            43|
|2020-01-23 23:00:00| 129| 304.1300000000001|            22|
|2020-01-28 15:00:00| 129|            208.23|             9|
|2020-01-02 08:00:00|  74|1187.4199999999987|            96|
|2020-01-06 16:00:00| 129|250.42000000000002|            15|
|2020-01-07 10:00:00| 171|             38.98|             2|
|2020-01-09 15:00:00| 260| 242.6800000000001|            19|
|2020-01-16 20:00:00|  82| 424.1900000000001|            27|
|2020-01-21 16:00:00|  33| 917.3599999999994|            38|
|2020-01-08 17:00:00| 140|            313.81|             8|
|2020-01-22 08:00:00| 244| 285.3999999999999|            14|
|2020-01-30 14:00:00|  74| 738.6799999999996|            49|
|2020-01-03 07:00:00|  7

In [3]:
df_green_revenue.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[_groupingexpression#46, PULocationID#5], functions=[sum(total_amount#16), count(1)])
   +- Exchange hashpartitioning(_groupingexpression#46, PULocationID#5, 200), ENSURE_REQUIREMENTS, [plan_id=95]
      +- HashAggregate(keys=[_groupingexpression#46, PULocationID#5], functions=[partial_sum(total_amount#16), partial_count(1)])
         +- Project [PULocationID#5, total_amount#16, date_trunc(hour, lpep_pickup_datetime#1, Some(America/New_York)) AS _groupingexpression#46]
            +- Filter (isnotnull(lpep_pickup_datetime#1) AND (lpep_pickup_datetime#1 >= 2020-01-01 00:00:00))
               +- FileScan parquet [lpep_pickup_datetime#1,PULocationID#5,total_amount#16] Batched: true, DataFilters: [isnotnull(lpep_pickup_datetime#1), (lpep_pickup_datetime#1 >= 2020-01-01 00:00:00)], Format: Parquet, Location: InMemoryFileIndex(19 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/pq/..., Part

In [4]:
df_green_revenue.write.mode("overwrite").parquet("data/report/revenue/green_aqe_on")

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.sql(query).write.mode("overwrite").parquet("data/report/revenue/green_aqe_off")
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [6]:
spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.sql(query).write.mode("overwrite").parquet("data/report/revenue/green_4")
spark.conf.set("spark.sql.shuffle.partitions", "200")

In [7]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.sql(query).write.mode("overwrite").parquet("data/report/revenue/green_4_aqe_off")
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [8]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("joins")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

26/11/18 22:34:28 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [20]:
spark.read.parquet("data/pq/green/*/*").createOrReplaceTempView("green")
spark.read.parquet("data/pq/yellow/*/*").createOrReplaceTempView("yellow")

for color, ts in [("green", "lpep_pickup_datetime"), ("yellow", "tpep_pickup_datetime")]:
    spark.sql(f"""
        SELECT
            date_trunc('hour', {ts}) AS hour,
            PULocationID AS zone,
            SUM(total_amount) AS amount,
            COUNT(1) AS number_records
        FROM {color}
        WHERE {ts} >= '2020-01-01' AND {ts} < '2021-01-01'
        GROUP BY 1, 2
    """).write.mode("overwrite").parquet(f"data/report/revenue/{color}_hourly")
    print(color, "done")

26/11/18 22:44:19 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:401)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:209)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

green done


yellow done


In [21]:
df_green = (
    spark.read.parquet("data/report/revenue/green_hourly")
    .withColumnRenamed("amount", "green_amount")
    .withColumnRenamed("number_records", "green_number_records")
)
df_yellow = (
    spark.read.parquet("data/report/revenue/yellow_hourly")
    .withColumnRenamed("amount", "yellow_amount")
    .withColumnRenamed("number_records", "yellow_number_records")
)

In [23]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

df_join = df_green.join(df_yellow, on=["hour", "zone"], how="outer")
df_join.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [coalesce(hour#577, hour#584) AS hour#591, coalesce(zone#578, zone#585) AS zone#592, green_amount#582, green_number_records#583L, yellow_amount#589, yellow_number_records#590L]
   +- SortMergeJoin [hour#577, zone#578], [hour#584, zone#585], FullOuter
      :- Sort [hour#577 ASC NULLS FIRST, zone#578 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(hour#577, zone#578, 200), ENSURE_REQUIREMENTS, [plan_id=1204]
      :     +- Project [hour#577, zone#578, amount#579 AS green_amount#582, number_records#580L AS green_number_records#583L]
      :        +- FileScan parquet [hour#577,zone#578,amount#579,number_records#580L] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/repo..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<hour:timestamp,zone:int,amount:double,number_records:bigint>
      +- So

In [24]:
df_join.write.mode("overwrite").parquet("data/report/revenue/total")

In [31]:
df_zones = spark.read.option("header", True).csv("data/zones/")



In [32]:
df_zones = (
    spark.read.option("header", True).csv("data/zones/")
    .withColumnRenamed("LocationID", "location_id")
    .withColumnRenamed("Borough", "borough")
    .withColumnRenamed("Zone", "zone_name")
    .withColumn("location_id", F.col("location_id").cast("int"))
)

df_revenue = spark.read.parquet("data/report/revenue/total")

df_result = (
    df_revenue
    .join(df_zones, df_revenue.zone == df_zones.location_id)
    .drop("location_id")
)
df_result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [hour#750, zone#751, green_amount#752, green_number_records#753L, yellow_amount#754, yellow_number_records#755L, borough#747, zone_name#748, service_zone#744]
   +- SortMergeJoin [zone#751], [location_id#749], Inner
      :- Sort [zone#751 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(zone#751, 200), ENSURE_REQUIREMENTS, [plan_id=1814]
      :     +- Filter isnotnull(zone#751)
      :        +- FileScan parquet [hour#750,zone#751,green_amount#752,green_number_records#753L,yellow_amount#754,yellow_number_records#755L] Batched: true, DataFilters: [isnotnull(zone#751)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/repo..., PartitionFilters: [], PushedFilters: [IsNotNull(zone)], ReadSchema: struct<hour:timestamp,zone:int,green_amount:double,green_number_records:bigint,yellow_amount:doub...
      +- Sort [location_id#749 ASC NULLS F

In [30]:
df_result.write.mode("overwrite").parquet("data/report/revenue/total_with_zones")
df_result.select("hour", "borough", "zone_name", "green_amount", "yellow_amount").show(5)

+-------------------+-------+--------------+------------+-------------+
|               hour|borough|     zone_name|green_amount|yellow_amount|
+-------------------+-------+--------------+------------+-------------+
|2020-01-01 07:00:00|    EWR|Newark Airport|        NULL|         0.31|
|2020-01-01 09:00:00|    EWR|Newark Airport|        NULL|       175.26|
|2020-01-01 10:00:00|    EWR|Newark Airport|        NULL|        100.6|
|2020-01-01 11:00:00|    EWR|Newark Airport|        NULL|         90.3|
|2020-01-01 12:00:00|    EWR|Newark Airport|        NULL|       266.66|
+-------------------+-------+--------------+------------+-------------+
only showing top 5 rows


In [33]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'-1'

In [34]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")   # 10 MB, the default

df_result_bc = (
    df_revenue
    .join(df_zones, df_revenue.zone == df_zones.location_id)
    .drop("location_id")
)
df_result_bc.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [hour#750, zone#751, green_amount#752, green_number_records#753L, yellow_amount#754, yellow_number_records#755L, borough#747, zone_name#748, service_zone#744]
   +- BroadcastHashJoin [zone#751], [location_id#749], Inner, BuildRight, false, false
      :- Filter isnotnull(zone#751)
      :  +- FileScan parquet [hour#750,zone#751,green_amount#752,green_number_records#753L,yellow_amount#754,yellow_number_records#755L] Batched: true, DataFilters: [isnotnull(zone#751)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/repo..., PartitionFilters: [], PushedFilters: [IsNotNull(zone)], ReadSchema: struct<hour:timestamp,zone:int,green_amount:double,green_number_records:bigint,yellow_amount:doub...
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, true] as bigint)),false), [plan_id=1846]
         +- Project [cast(LocationID#741 as i

In [35]:
df_result_bc.write.mode("overwrite").parquet("data/report/revenue/total_with_zones")

26/11/19 09:24:24 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 35094300 ms exceeds timeout 120000 ms
26/11/19 09:24:25 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/19 09:24:34 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:143)
	at org.apache.spark.sto